# E05:
look up and use F.cross_entropy instead. You should achieve the same result. Can you think of why we'd prefer to use F.cross_entropy instead?


## Why `F.cross_entropy(logits, ys)` equals the manual version

The manual version (from E04) computed, for one sample with logit row $z = (z_0, \ldots, z_{26})$:

```python
counts = logits.exp()                          # exp(z_k)
probs  = counts / counts.sum(1, keepdim=True)   # softmax
loss   = -probs[arange, ys].log().mean()        # -log(a_c), averaged over N
```

where the softmax is

$$a_k = \frac{\exp(z_k)}{\sum_j \exp(z_j)},$$

so the per-sample loss is

$$l_n = -\log a_c, \qquad a_c = \frac{\exp(z_c)}{\sum_j \exp(z_j)}, \qquad c = \texttt{ys}[n].$$

### What the PyTorch docs define

The `CrossEntropyLoss` formula (class-indices form) is

$$l_n = - w_{y_n} \, \log\!\left( \frac{\exp\big(x_{n,\, y_n}\big)}{\sum_{c=1}^{C} \exp\big(x_{n,\, c}\big)} \right).$$

- $x_{n,\,c}$ is the input **logit** for sample $n$, class $c$. So $x_{n,\,y_n}$ is the logit of sample $n$ at its **correct class** $y_n$. In our code, $x_{n,\,y_n} = \texttt{logits}[n,\, \texttt{ys}[n]]$, and the ratio $\dfrac{\exp(x_{n,\,y_n})}{\sum_c \exp(x_{n,\,c})}$ is exactly the softmax of row $n$ read off at the correct class, i.e. $\texttt{probs}[n,\, \texttt{ys}[n]]$.
- $w_{y_n}$ is the optional per-class `weight` (a length-$C$ tensor for class imbalance). We pass no `weight`, so every $w_{y_n} = 1$ and the term vanishes.

### So they are identical

With $w_{y_n} = 1$, the docs' formula becomes

$$l_n = -\log\big(\texttt{probs}[n,\, \texttt{ys}[n]]\big),$$

which is literally the manual line. `F.cross_entropy` just fuses softmax $\to$ log $\to$ gather-target $\to$ negate $\to$ mean into one call.

The only difference is the *numerical path*: `F.cross_entropy` uses the algebraically equivalent log-sum-exp form

$$l_n = -x_{n,\, y_n} + \log \sum_{c} \exp\big(x_{n,\, c}\big),$$

and subtracts each row's max before $\exp$, avoiding the overflow/underflow that raw `logits.exp()` can hit. Same math, more stable and faster — which is why we prefer it.

**Two gotchas:** (1) pass *logits*, not `probs` (it applies softmax internally); (2) L2 regularization is not included — add `+ 0.1 * (W**2).mean()` separately if needed.


In [3]:
# import
import torch
import torch.nn.functional as F

In [4]:
# split data set
words = open('name.txt', 'r').read().splitlines()
train_words = words

# build stoi and itos
chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}

In [5]:
# bigram model
def parse_bigram_data(words):
    # create the training set of bigrams
    xs, ys = [], []
    for w in words:
        chs = ["."] + list(w) + ["."]
        for ch1, ch2 in zip(chs, chs[1:]):
            ix1 = stoi[ch1]
            ix2 = stoi[ch2]
            xs.append(ix1)
            ys.append(ix2)

    xs = torch.tensor(xs)
    ys = torch.tensor(ys)

    return xs, ys

def train_bigram_model(n_iter=100):
    xs, ys = parse_bigram_data(train_words)

    # initialize the network
    g = torch.Generator().manual_seed(2147483647)
    W = torch.randn((27, 27), generator=g, requires_grad=True)

    # gradient descent
    for k in range(n_iter):
        # forward pass
        logits = W[xs]
        loss = F.cross_entropy(logits, ys)
        if k == n_iter - 1:
            print("train loss: ", loss.item())

        # backward pass
        W.grad = None  # set to zero the gardient
        loss.backward()

        # update
        W.data += -50 * W.grad

    return W


@torch.no_grad()
def evaluate_bigram_model(W, evaluate_words):
    xs, ys = parse_bigram_data(evaluate_words)
    logits = W[xs]  # the 1-hot @ W is just a row select, so index W directly
    loss = F.cross_entropy(logits, ys)

    return loss.item()

In [6]:
# evalute and compare
W_bi = train_bigram_model(n_iter=100)

train loss:  2.47287654876709
